In [1]:
using ArgParse
using Printf
using Dates
using JLD2
using ITensors
using ITensorMPS
using LinearAlgebra

const ROOT = normpath(joinpath(@__DIR__, ".."))
include(joinpath(ROOT, "QCSB", "QCSB.jl"))

include(joinpath(ROOT, "src", "circuit.jl"))
include(joinpath(ROOT, "src", "tci.jl"))

tci (generic function with 1 method)

In [ ]:
using ITensors

# scalar Z = ⟨+...+|ψ⟩, without building |+...+⟩ as an MPS
function overlap_plus(ψ::MPS)
    sites = siteinds(ψ)
    E = ITensor(1.0)
    for j in 1:length(sites)
        s = sites[j]

        # <+| = (⟨0|+⟨1|)/√2 or (⟨Up|+⟨Dn|)/√2 depending on siteset
        # We'll try common labels; adjust if needed.
        v0 = ITensors.state(s, "0")
        v1 = ITensors.state(s, "1")
        bra = dag(v0 + v1)

        E *= bra * ψ[j]
    end
    return scalar(E)
end

function normalize(state::DiagonalStateMPS)
    ψ = state.mps
    L = length(siteinds(ψ))

    Z = overlap_plus(ψ)               # ⟨+...+|ψ⟩
    λ = log(Z) / L                    # per-site log amplitude (use logabs if needed)
    s = exp(-λ)

    for j in 1:L
        ψ[j] *= s
    end
    return DiagonalStateMPS(ψ)
end


normalize (generic function with 1 method)

In [79]:
L = 10
state = neel_state(DiagonalStateMPS, L; conserve_qns=true)
state = normalize(state)

DiagonalStateMPS(MPS(10))

In [ ]:
function normalize(state::DiagonalStateMPS)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)
    λ = (loginner(MPS(sites, i -> "+"), ψ) + (L/2)*log(2)) / L
    s = exp(-λ)
    for j in 1:L
        ψ[j] *= s
    end
    return DiagonalStateMPS(ψ)
end

In [98]:
function simple_circuit(state::DiagonalStateMPS, L::Int, T::Int, p::Float64; cutoff=1e-8, maxdim=200)
    SWAPn1 = decoherence_layer(state, SWAP, p, 1:2:L-1)
    SWAPn2 = decoherence_layer(state, SWAP, p, 2:2:L-1)

    for t in 1:T
        state = apply(SWAPn1, state; cutoff=cutoff, maxdim=maxdim)
        state = apply(SWAPn2, state; cutoff=cutoff, maxdim=maxdim)
        state = normalize(state)
        # state /= norm(state.mps)
        truncate!(state; cutoff=cutoff, maxdim=maxdim)
    end

    return state
end

simple_circuit (generic function with 1 method)

In [83]:
L = 10
state = neel_state(DiagonalStateMPS, L; conserve_qns=true)

SWAPn1 = decoherence_layer(state, SWAP, 0.1, 1:2:L)
SWAPn2 = decoherence_layer(state, SWAP, 0.1, 2:3:L)

state = apply(SWAPn1, state)
state = apply(SWAPn2, state)
state /= norm(state.mps)
truncate!(state; cutoff=1e-8, maxdim=200)

state = apply(SWAPn1, state)
state = apply(SWAPn2, state)
truncate!(state; cutoff=1e-8, maxdim=200)

state = apply(SWAPn1, state)
state = apply(SWAPn2, state)
truncate!(state; cutoff=1e-8, maxdim=200)

DiagonalStateMPS(MPS(10))

In [100]:
L = 100
state = neel_state(DiagonalStateMPS, L; conserve_qns=true)

state = simple_circuit(state, L, L, 0.1)

DiagonalStateMPS(MPS(100))

In [103]:
measure(state, PauliZ, 1.0, 1)

ErrorException: Fluxes not all equal

In [32]:
ψ = state.mps
adder1 = adder_itensor(only(inds(ψ[1],"Site")), only(inds(ψ[2],"Site")))
cum1 = adder1 * ψ[1] * ψ[2]

adder2 = adder_itensor(only(inds(cum1,"Site")), only(inds(ψ[3],"Site")))
cum2 = adder2 * cum1 * ψ[3]

ITensor ord=2 (dim=4|id=83|"Site") (dim=2|id=939|"Link,l=3")
NDTensors.Dense{Float64, Vector{Float64}}

In [13]:
function adder_itensor(ind1::Index{Int}, ind2::Index{Int})
    d1 = dim(ind1)
    d2 = dim(ind2)
    dout = d1 + d2 - 1
    ind_out = Index(dout, "Site")
    K = ITensor(ind_out, ind1, ind2)
    for a in 1:d1
        for b in 1:d2
            c = a + b - 1
            K[ind_out => c, ind1 => a, ind2 => b] = 1.0
        end
    end
    return K
end

adder_itensor (generic function with 1 method)

In [ ]:
"""
Adder tensor K : H(d1) ⊗ H(d2) → H(dout)
with K|a⟩|b⟩ = |a+b⟩ (0-based labels).
"""
function adder_itensor(d1::Int, d2::Int; dout::Int=d1+d2-1,
                      tags1="s1", tags2="s2", tagsout="sout")
  @assert dout ≥ d1 + d2 - 1

  s1   = Index(d1, tags1)
  s2   = Index(d2, tags2)
  sout = Index(dout, tagsout)

  # K has one ket index (sout) and two bra indices (dag(s1), dag(s2))
  K = ITensor(sout, dag(s1), dag(s2))

  # Fill: K[sout=c, s1=a, s2=b] = 1 if c = a+b
  for a in 0:d1-1
    for b in 0:d2-1
      c = a + b
      K[sout => c+1, s1 => a+1, s2 => b+1] = 1.0
    end
  end

  return K, s1, s2, sout
end

K, s1, s2, sout = adder_itensor(3, 5)  # dout defaults to d1+d2-1 = 7
